# Masked Multi-Task Fingerprint Model

## Scientific objective
Train a shared fingerprint encoder with endpoint-specific heads using masked BCE so unavailable labels never become negatives.

## Inputs
- Global molecule split
- Morgan matrix
- Endpoint label matrix

## Expected outputs
- `models/multitask/fingerprint_multitask.pt`
- `results/metrics/multitask_fingerprint.csv`
- transfer table

## Dependencies
PyTorch

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
One global molecule/scaffold split is used. Loss normalization includes only observed molecule–endpoint pairs.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Multi-task sharing can cause negative transfer; every endpoint is compared against its single-task counterpart.

## Next notebook
[14_multitask_gnn_model.ipynb](./14_multitask_gnn_model.ipynb)


In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})


{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
import torch
from torch.utils.data import DataLoader
from toxicity_screening.datasets import MaskedMultiTaskArrayDataset
from toxicity_screening.neural_models import MultiTaskFingerprintMLP
from toxicity_screening.training import train_masked_multitask_model, resolve_device
from toxicity_screening.losses import positive_class_weights
from toxicity_screening.metrics import binary_metrics

endpoints = list(CONFIGS["endpoints"]["endpoints"])
archive = np.load(ROOT / "data/processed/morgan_features.npz")
X = archive["X"].astype(np.float32)
ids = archive["molecule_id"].astype(str)

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)

labels = (
    records.pivot_table(
        index="molecule_id",
        columns="endpoint",
        values="label",
        aggfunc="first",
    )
    .reindex(index=ids, columns=endpoints)
)
splits = (
    records[["molecule_id", "scaffold_split"]]
    .drop_duplicates("molecule_id")
    .set_index("molecule_id")
    .reindex(ids)
    .scaffold_split
)
idx = {
    p: np.where(splits.to_numpy() == p)[0]
    for p in ["train", "validation", "test"]
}
loaders = {
    p: DataLoader(
        MaskedMultiTaskArrayDataset(
            X[i],
            labels.to_numpy(float)[i],
        ),
        batch_size=CONFIGS["training_config"]["batch_size"],
        shuffle=p == "train",
    )
    for p, i in idx.items()
    if p != "test"
}
model = MultiTaskFingerprintMLP(
    X.shape[1],
    len(endpoints),
    **CONFIGS["model_config"]["neural"]["multitask_mlp"],
)
train_labels = torch.tensor(
    np.nan_to_num(
        labels.to_numpy(float)[idx["train"]],
        nan=0.0,
    )
)
train_mask = torch.tensor(
    ~np.isnan(labels.to_numpy(float)[idx["train"]]),
    dtype=torch.float32,
)
pos_weights = positive_class_weights(train_labels, train_mask)
counts = train_mask.sum(0)
task_weights = 1 / torch.sqrt(counts.clamp_min(1))
task_weights = task_weights / task_weights.mean()
result = train_masked_multitask_model(
    model,
    loaders["train"],
    loaders["validation"],
    epochs=PROFILE_CONFIG["max_epochs"],
    patience=PROFILE_CONFIG["patience"],
    learning_rate=CONFIGS["training_config"]["optimizer"]["learning_rate"],
    weight_decay=CONFIGS["training_config"]["optimizer"]["weight_decay"],
    gradient_clip_norm=CONFIGS["training_config"]["gradient_clip_norm"],
    positive_weights=pos_weights,
    task_weights=task_weights,
    checkpoint_path=ROOT / "models/multitask/fingerprint_multitask.pt",
)
pd.DataFrame(result.history).to_csv(
    ROOT / "results/metrics/multitask_fingerprint_history.csv",
    index=False,
)
model.eval()
device = resolve_device()
model.to(device)
with torch.no_grad():
    probs = torch.sigmoid(
        model(
            torch.as_tensor(
                X[idx["test"]],
                dtype=torch.float32,
                device=device,
            )
        )
    ).cpu().numpy()

rows = []
yall = labels.to_numpy(float)[idx["test"]]
for k, endpoint in enumerate(endpoints):
    obs = ~np.isnan(yall[:, k])
    rows.append(
        {
            "endpoint": endpoint,
            "model": "multitask_fingerprint_mlp",
            **{
                a: b
                for a, b in binary_metrics(
                    yall[obs, k].astype(int),
                    probs[obs, k],
                ).items()
                if a != "confusion_matrix"
            },
        }
    )
mt_metrics = pd.DataFrame(rows)
mt_metrics.to_csv(
    ROOT / "results/metrics/multitask_fingerprint.csv",
    index=False,
)
display(mt_metrics)


,endpoint,model,n,positive_prevalence,threshold,roc_auc,pr_auc,mcc,accuracy,balanced_accuracy,...,f1,brier,ece,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp
0,herg_blockade,multitask_fingerprint_mlp,1883,0.507169,0.5,0.795906,0.824362,0.425582,0.708975,0.710149,...,0.686499,0.183080,0.043956,0.540304,0.589529,0.675199,735,193,355,600
1,ames_mutagenicity,multitask_fingerprint_mlp,1120,0.547321,0.5,0.823849,0.856217,0.489362,0.743750,0.745783,...,0.755745,0.175390,0.061942,0.528797,0.704731,0.748476,389,118,169,444
2,SR-p53,multitask_fingerprint_mlp,986,0.085193,0.5,0.752745,0.302387,0.149687,0.464503,0.631744,...,0.209581,0.327252,0.476228,0.893397,0.011905,0.125461,388,514,14,70
3,SR-ATAD5,multitask_fingerprint_mlp,1027,0.064265,0.5,0.797449,0.215451,0.184464,0.478092,0.685846,...,0.185410,0.313929,0.478520,0.859842,0.000000,0.160606,430,531,5,61
4,SR-ARE,multitask_fingerprint_mlp,847,0.205431,0.5,0.769026,0.501538,0.304201,0.563164,0.684630,...,0.455882,0.269027,0.361548,0.740316,0.126437,0.359897,322,351,19,155
5,SR-MMP,multitask_fingerprint_mlp,842,0.220903,0.5,0.839542,0.612072,0.383763,0.619952,0.729134,...,0.518072,0.243749,0.331666,0.690960,0.188172,0.465839,350,306,14,172


In [3]:
from toxicity_screening.evaluation import endpoint_transfer

single = pd.read_csv(
    ROOT / "results/metrics/single_task_mlp.csv"
)
transfer = []
for endpoint in mt_metrics.endpoint:
    s = float(
        single.loc[
            single.endpoint == endpoint,
            "pr_auc",
        ].iloc[0]
    )
    m = float(
        mt_metrics.loc[
            mt_metrics.endpoint == endpoint,
            "pr_auc",
        ].iloc[0]
    )
    transfer.append(
        {
            "endpoint": endpoint,
            "metric": "pr_auc",
            **endpoint_transfer(s, m),
        }
    )
transfer = pd.DataFrame(transfer)
transfer.to_csv(
    ROOT / "results/ablations/multitask_transfer_fingerprint.csv",
    index=False,
)
display(transfer)


,endpoint,metric,single_task,multitask,transfer,category
0,herg_blockade,pr_auc,0.821754,0.824362,0.002607,neutral
1,ames_mutagenicity,pr_auc,0.879032,0.856217,-0.022816,negative
2,SR-p53,pr_auc,0.318745,0.302387,-0.016358,negative
3,SR-ATAD5,pr_auc,0.175134,0.215451,0.040317,positive
4,SR-ARE,pr_auc,0.501459,0.501538,0.000079,neutral
5,SR-MMP,pr_auc,0.635222,0.612072,-0.023150,negative


### Completion gate
Confirm that the declared artifacts exist before continuing to `14_multitask_gnn_model.ipynb`.